In [1]:
# ---------- Cellule 0 : configuration et chargement de Green renommé ----------
import os, time, shutil, warnings
import numpy as np
import pandas as pd
from IPython.display import display
warnings.filterwarnings("ignore")

RENOM_DIR = os.path.join(os.getcwd(), "renamed_data")
CLEAN_DIR = os.path.join(os.getcwd(), "clean_data")
os.makedirs(CLEAN_DIR, exist_ok=True)

TYPES = {"zone_depart": "float32", "zone_arrivee": "float32", "distance": "float32", "prix": "float32",
         "pourboire": "float32", "paiement": "float32", "frais_cbd": "float32"}
DATES = ["date_debut", "date_fin"]

t0 = time.time()
green = pd.read_csv(os.path.join(RENOM_DIR, "green_renomme.csv"), dtype=TYPES)
for c in DATES:
    green[c] = pd.to_datetime(green[c], format="ISO8601", errors="coerce")
N_DEPART = len(green)
print(f"Green chargé : {N_DEPART:,} lignes, {green.shape[1]} colonnes – "
      f"{green.memory_usage(deep=True).sum()/1e9:.2f} Go – {time.time()-t0:,.0f} s")
display(green.head())

Green chargé : 209,194 lignes, 9 colonnes – 0.01 Go – 1 s


,date_debut,date_fin,zone_depart,zone_arrivee,distance,prix,pourboire,paiement,frais_cbd
0,2024-02-01 00:15:53,2024-02-01 00:24:20,75.0,161.0,2.77,13.500000,3.75,1.0,NaN
1,2024-01-31 22:59:22,2024-01-31 23:27:14,43.0,242.0,7.76,36.599998,0.00,2.0,NaN
2,2024-02-01 00:30:29,2024-02-01 00:35:32,75.0,238.0,1.03,7.900000,2.60,1.0,NaN
3,2024-01-31 23:56:42,2024-02-01 00:06:53,80.0,17.0,1.36,11.400000,4.17,1.0,NaN
4,2024-02-01 00:31:14,2024-02-01 00:31:16,95.0,264.0,0.00,50.000000,1.00,1.0,NaN


In [3]:
# ---------- Cellule 1 : description générale de Green ----------
print("=== info() ===")
green.info(memory_usage="deep", show_counts=True)

nb_lignes = len(green)
nb_doublons = int(green.duplicated().sum())

resume = pd.DataFrame({
    "type":                  green.dtypes.astype(str),
    "nb_non_null":           green.notna().sum(),
    "nb_null":               green.isna().sum(),
    "pct_null":              (green.isna().mean() * 100).round(3),
    "nb_valeurs_distinctes": green.nunique(),
    "exemple":               green.iloc[0],
})
print(f"\nRésumé par colonne ({nb_lignes:,} lignes, {green.shape[1]} colonnes) ")
display(resume)

print(f"\nLignes en double (les 9 colonnes identiques) : {nb_doublons:,} ({nb_doublons/nb_lignes:.3%})")
print(f"Lignes avec au moins une valeur manquante  : {int(green.isna().any(axis=1).sum()):,}")

print("\ndescribe() des colonnes numériques ")
display(green.describe(include="number", percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).T.round(2))

print("\nPériode couverte ")
print(f"date_debut : du {green['date_debut'].min()} au {green['date_debut'].max()}")
print(f"date_fin   : du {green['date_fin'].min()} au {green['date_fin'].max()}")

=== info() ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 209194 entries, 0 to 209193
Data columns (total 9 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   date_debut    209194 non-null  datetime64[ns]
 1   date_fin      209194 non-null  datetime64[ns]
 2   zone_depart   209194 non-null  float32       
 3   zone_arrivee  209194 non-null  float32       
 4   distance      209194 non-null  float32       
 5   prix          209194 non-null  float32       
 6   pourboire     209194 non-null  float32       
 7   paiement      198103 non-null  float32       
 8   frais_cbd     96172 non-null   float32       
dtypes: datetime64[ns](2), float32(7)
memory usage: 8.8 MB

Résumé par colonne (209,194 lignes, 9 colonnes) 


,type,nb_non_null,nb_null,pct_null,nb_valeurs_distinctes,exemple
date_debut,datetime64[ns],209194,0,0.000,205040,2024-02-01 00:15:53
date_fin,datetime64[ns],209194,0,0.000,205049,2024-02-01 00:24:20
zone_depart,float32,209194,0,0.000,242,75.0
zone_arrivee,float32,209194,0,0.000,252,161.0
distance,float32,209194,0,0.000,2540,2.77
prix,float32,209194,0,0.000,3986,13.5
pourboire,float32,209194,0,0.000,1965,3.75
paiement,float32,198103,11091,5.302,5,1.0
frais_cbd,float32,96172,113022,54.027,3,NaN



Lignes en double (les 9 colonnes identiques) : 12 (0.006%)
Lignes avec au moins une valeur manquante  : 117,100

describe() des colonnes numériques 


,count,mean,std,min,1%,25%,50%,75%,99%,max
zone_depart,209194.0,96.21,57.01,1.00,7.0,74.00,75.0,97.00,260.00,265.00
zone_arrivee,209194.0,140.26,76.75,1.00,7.0,74.00,138.0,225.00,264.00,265.00
distance,209194.0,15.45,862.03,0.00,0.0,1.11,1.8,3.13,15.73,147993.11
prix,209194.0,17.20,14.80,-470.60,2.9,9.30,13.5,19.80,70.00,955.70
pourboire,209194.0,2.42,3.21,-20.00,0.0,0.00,2.0,3.65,12.90,222.00
paiement,198103.0,1.29,0.49,1.00,1.0,1.00,1.0,2.00,2.00,5.00
frais_cbd,96172.0,0.07,0.21,-0.75,0.0,0.00,0.0,0.00,0.75,0.75



Période couverte 
date_debut : du 2008-12-31 23:02:24 au 2025-04-01 23:41:29
date_fin   : du 2008-12-31 23:02:30 au 2025-04-01 23:41:50


In [4]:
# ---------- Cellule 2 : suppression des doublons ----------
avant = len(green)
masque_doublons = green.duplicated(keep="first")
nb_doublons = int(masque_doublons.sum())

print(f"Doublons exacts (les 9 colonnes identiques) : {nb_doublons:,} ({nb_doublons/avant:.3%})")
if nb_doublons:
    exemple = green[green.duplicated(keep=False)].sort_values(["date_debut", "zone_depart"]).head(6)
    print("Exemple de lignes répétées :")
    display(exemple)

green = green[~masque_doublons].reset_index(drop=True)
print(f"Avant : {avant:,} lignes -> après : {len(green):,} lignes ({avant - len(green):,} supprimées)")

Doublons exacts (les 9 colonnes identiques) : 12 (0.006%)
Exemple de lignes répétées :


,date_debut,date_fin,zone_depart,zone_arrivee,distance,prix,pourboire,paiement,frais_cbd
30176,2024-02-18 01:44:59,2024-02-18 01:45:07,255.0,255.0,0.02,0.0,0.0,2.0,NaN
30177,2024-02-18 01:44:59,2024-02-18 01:45:07,255.0,255.0,0.02,0.0,0.0,2.0,NaN
31586,2024-02-19 00:02:42,2024-02-19 16:34:55,138.0,28.0,9.76,0.0,0.0,2.0,NaN
31587,2024-02-19 00:02:42,2024-02-19 16:34:55,138.0,28.0,9.76,0.0,0.0,2.0,NaN
41677,2024-02-25 04:31:56,2024-02-25 04:32:32,256.0,256.0,0.00,0.0,0.0,3.0,NaN
41678,2024-02-25 04:31:56,2024-02-25 04:32:32,256.0,256.0,0.00,0.0,0.0,3.0,NaN


Avant : 209,194 lignes -> après : 209,182 lignes (12 supprimées)


In [5]:
# ---------- Cellule 3 : valeurs vides ----------
annee = green["date_debut"].dt.year
verif = pd.DataFrame({
    "nb_courses":  annee.value_counts(),
    "nb_null_cbd": green["frais_cbd"].isna().groupby(annee).sum(),
}).fillna(0).astype(int).sort_index()
verif["pct_null_cbd"] = (verif["nb_null_cbd"] / verif["nb_courses"] * 100).round(2)
print("Vides de frais_cbd par année :")
display(verif)

avant_cbd = int(green["frais_cbd"].isna().sum())
green["frais_cbd"] = green["frais_cbd"].fillna(0)
print(f"frais_cbd : {avant_cbd:,} vides -> 0 (péage inexistant avant le 5 janvier 2025)")

avant_pai = int(green["paiement"].isna().sum())
green["paiement"] = green["paiement"].fillna(5)
print(f"paiement  : {avant_pai:,} vides -> code 5 (inconnu)")

print("\nVides restants par colonne :")
display(green.isna().sum().to_frame("nb_null"))

Vides de frais_cbd par année :


,nb_courses,nb_null_cbd,pct_null_cbd
date_debut,,,
2008,2,2,100.00
2024,111027,111027,100.00
2025,98153,1988,2.03


frais_cbd : 113,017 vides -> 0 (péage inexistant avant le 5 janvier 2025)
paiement  : 11,091 vides -> code 5 (inconnu)

Vides restants par colonne :


,nb_null
date_debut,0
date_fin,0
zone_depart,0
zone_arrivee,0
distance,0
prix,0
pourboire,0
paiement,0
frais_cbd,0


Elle vérifie que les vides de frais_cbd viennent de 2024 et les remplace par 0, met les paiements vides au code 5 (inconnu), puis affiche les vides restants.
Effet sur les données : plus aucun vide dans ces deux colonnes, sans perte de lignes.

In [7]:
# ---------- Cellule 4 : filtre sur la période (fév-mars 2024 et fév-mars 2025) ----------
MOIS_ETUDE = ["2024-02", "2024-03", "2025-02", "2025-03"]

avant = len(green)
mois = green["date_debut"].dt.strftime("%Y-%m")
dans_periode = mois.isin(MOIS_ETUDE)

print("Courses hors période :")
display(mois[~dans_periode].value_counts().sort_index().to_frame("nb_courses"))

green = green[dans_periode].reset_index(drop=True)
print(f"Avant : {avant:,} lignes , après : {len(green):,} lignes ({avant - len(green):,} supprimées)")
print(f"Période gardée : du {green['date_debut'].min()} au {green['date_debut'].max()}")

print("Courses gardées par mois :")
display(green["date_debut"].dt.strftime("%Y-%m").value_counts().sort_index().to_frame("nb_courses"))

Courses hors période :


,nb_courses
date_debut,


Avant : 209,163 lignes , après : 209,163 lignes (0 supprimées)
Période gardée : du 2024-02-01 00:00:54 au 2025-03-31 23:54:20
Courses gardées par mois :


,nb_courses
date_debut,
2024-02,53575
2024-03,57445
2025-02,46605
2025-03,51538


Elle garde uniquement février et mars 2024, et février et mars 2025.
Effet sur les données : les courses aux dates fausses sont supprimées.

In [9]:
# ---------- Cellule 5 : calcul et contrôle de la durée ----------
DUREE_MIN, DUREE_MAX = 1, 180          # minutes

green["duree_min"] = ((green["date_fin"] - green["date_debut"]).dt.total_seconds() / 60).astype("float32")

print("Distribution de la durée (minutes) avant contrôle :")
display(green["duree_min"].describe(percentiles=[0.001, 0.01, 0.5, 0.99, 0.999]).to_frame("duree_min").T.round(2))

d = green["duree_min"]
print(f"Durée négative         : {int((d < 0).sum()):,}")
print(f"Durée entre 0 et 1 min : {int(((d >= 0) & (d < DUREE_MIN)).sum()):,}")
print(f"Durée > {DUREE_MAX} min        : {int((d > DUREE_MAX).sum()):,}")

invalide = (d < DUREE_MIN) | (d > DUREE_MAX)
green.loc[invalide, "duree_min"] = np.nan
print(f"\nDurées mises à vide : {int(invalide.sum()):,} ({invalide.mean():.2%}) – ")
print(f"Nombre de lignes : {len(green):,} ")

Distribution de la durée (minutes) avant contrôle :


,count,mean,std,min,0.1%,1%,50%,99%,99.9%,max
duree_min,209163.0,19.28,82.01,-7.43,-0.37,0.05,11.62,65.42,1408.28,1500.77


Durée négative         : 518
Durée entre 0 et 1 min : 6,641
Durée > 180 min        : 916

Durées mises à vide : 8,075 (3.86%) – 
Nombre de lignes : 209,163 


les courses restent dans la table

Elle calcule duree_min et vide les durées hors de 1 à 180 minutes, en gardant les courses.
Effet sur les données : une colonne de plus, même nombre de lignes.

In [10]:
# ---------- Cellule 6 : contrôle des zones (1 à 263) ----------
ZONE_MIN, ZONE_MAX = 1, 263

avant = len(green)
dep_ok = green["zone_depart"].between(ZONE_MIN, ZONE_MAX)
arr_ok = green["zone_arrivee"].between(ZONE_MIN, ZONE_MAX)


display(green.loc[~dep_ok, "zone_depart"].value_counts(dropna=False).to_frame("nb_courses"))
display(green.loc[~arr_ok, "zone_arrivee"].value_counts(dropna=False).to_frame("nb_courses"))

green.loc[~arr_ok, "zone_arrivee"] = np.nan
green = green[dep_ok].reset_index(drop=True)
green["zone_depart"] = green["zone_depart"].astype("int16")
green["zone_arrivee"] = green["zone_arrivee"].astype("Int16")   # entier qui accepte les vides

print(f"Avant : {avant:,} lignes -> après : {len(green):,} lignes ({avant - len(green):,} supprimées)")
print(f"Zones de départ distinctes : {green['zone_depart'].nunique()}")

,nb_courses
zone_depart,
264.0,541
265.0,132


,nb_courses
zone_arrivee,
264.0,1883
265.0,661


Avant : 209,163 lignes -> après : 208,490 lignes (673 supprimées)
Zones de départ distinctes : 240


Zones d'arrivée invalides (valeur vidée, course gardée)
Zones de départ invalides (lignes supprimées)

Elle supprime les courses dont la zone de départ n’est pas entre 1 et 263, et vide la zone d’arrivée quand elle est inconnue.
Effet sur les données : quelques lignes en moins, et seules de vraies zones restent au départ.

In [11]:
# ---------- Cellule 7 : contrôle de la distance (miles) ----------
DIST_MAX = 100

print("Distribution de la distance avant contrôle :")
display(green["distance"].describe(percentiles=[0.01, 0.5, 0.99, 0.999]).to_frame("distance").T.round(2))

d = green["distance"]
print(f"Distance négative     : {int((d < 0).sum()):,}")
print(f"Distance égale à 0    : {int((d == 0).sum()):,}")
print(f"Distance > {DIST_MAX} miles : {int((d > DIST_MAX).sum()):,}")

invalide = (d <= 0) | (d > DIST_MAX)
green.loc[invalide, "distance"] = np.nan
print(f"\nDistances mises à vide : {int(invalide.sum()):,} ({invalide.mean():.2%})")
print(f"Nombre de lignes : {len(green):,} (inchangé)")

Distribution de la distance avant contrôle :


,count,mean,std,min,1%,50%,99%,99.9%,max
distance,208490.0,15.5,863.48,0.0,0.0,1.8,15.67,28.85,147993.11


Distance négative     : 0
Distance égale à 0    : 10,616
Distance > 100 miles : 82

Distances mises à vide : 10,698 (5.13%)
Nombre de lignes : 208,490 (inchangé)


Elle vide les distances qui ne sont pas strictement supérieures à 0 et au plus égales à 100 miles, en gardant les courses.
Effet sur les données : même nombre de lignes.

In [12]:
# ---------- Cellule 8 : contrôle du prix ($) ----------
PRIX_MAX = 500

print("Distribution du prix avant contrôle :")
display(green["prix"].describe(percentiles=[0.01, 0.5, 0.99, 0.999]).to_frame("prix").T.round(2))

p = green["prix"]
print(f"Prix négatif  : {int((p < 0).sum()):,}")
print(f"Prix égal à 0 : {int((p == 0).sum()):,}")
print(f"Prix > {PRIX_MAX} $  : {int((p > PRIX_MAX).sum()):,}")

print("\nPrix négatifs par moyen de paiement (3 = sans frais, 4 = litige) :")
display(green.loc[p < 0, "paiement"].value_counts().sort_index().to_frame("nb_courses"))

invalide = (p <= 0) | (p > PRIX_MAX)
green.loc[invalide, "prix"] = np.nan
print(f"\nPrix mis à vide : {int(invalide.sum()):,} ({invalide.mean():.2%})")
print(f"Nombre de lignes : {len(green):,} (inchangé)")

Distribution du prix avant contrôle :


,count,mean,std,min,1%,50%,99%,99.9%,max
prix,208490.0,17.17,14.7,-470.6,2.9,13.5,70.0,139.76,955.7


Prix négatif  : 666
Prix égal à 0 : 165
Prix > 500 $  : 8

Prix négatifs par moyen de paiement (3 = sans frais, 4 = litige) :


,nb_courses
paiement,
2.0,4
3.0,463
4.0,196
5.0,3



Prix mis à vide : 839 (0.40%)
Nombre de lignes : 208,490 (inchangé)


Elle vide les prix qui ne sont pas strictement supérieurs à 0 et au plus égaux à 500 $, et montre que les prix négatifs viennent des codes 3 et 4.
Effet sur les données : même nombre de lignes.

In [13]:
# ---------- Cellule 9 : contrôle du pourboire ($) ----------
PB_MAX = 200

print("Distribution du pourboire avant contrôle :")
display(green["pourboire"].describe(percentiles=[0.01, 0.5, 0.99, 0.999]).to_frame("pourboire").T.round(2))

pb = green["pourboire"]
print(f"Pourboire négatif : {int((pb < 0).sum()):,}")
print(f"Pourboire > {PB_MAX} $ : {int((pb > PB_MAX).sum()):,}")

print("\nPourboire moyen par moyen de paiement (1 = carte, 2 = espèces) :")
display(green.groupby("paiement")["pourboire"].agg(nb_courses="size", pourboire_moyen="mean").round(2))

invalide = (pb < 0) | (pb > PB_MAX)
green.loc[invalide, "pourboire"] = np.nan
print(f"\nPourboires mis à vide : {int(invalide.sum()):,} ({invalide.mean():.3%}) – les courses restent dans la table")
print(f"Nombre de lignes : {len(green):,} (inchangé)")

Distribution du pourboire avant contrôle :


,count,mean,std,min,1%,50%,99%,99.9%,max
pourboire,208490.0,2.42,3.15,-20.0,0.0,2.0,12.86,25.0,222.0


Pourboire négatif : 14
Pourboire > 200 $ : 1

Pourboire moyen par moyen de paiement (1 = carte, 2 = espèces) :


,nb_courses,pourboire_moyen
paiement,,
1.0,141693,3.35
2.0,53837,0.00
3.0,1368,0.00
4.0,510,0.01
5.0,11082,2.69



Pourboires mis à vide : 15 (0.007%) – les courses restent dans la table
Nombre de lignes : 208,490 (inchangé)


Elle vide les pourboires négatifs ou supérieurs à 200 $, et montre le pourboire moyen par moyen de paiement.
Effet sur les données : même nombre de lignes.

In [14]:
# ---------- Cellule 10 : contrôle du péage CBD ----------
annee = green["date_debut"].dt.year

print("Valeurs de frais_cbd par année AVANT correction :")
display(pd.crosstab(green["frais_cbd"], annee))

negatif = green["frais_cbd"] < 0
green.loc[negatif, "frais_cbd"] = 0
print(f"Péages négatifs ramenés à 0 : {int(negatif.sum()):,}")

print("\nValeurs de frais_cbd par année APRÈS correction :")
display(pd.crosstab(green["frais_cbd"], annee))
print(f"Nombre de lignes : {len(green):,} (inchangé)")

Valeurs de frais_cbd par année AVANT correction :


date_debut,2024,2025
frais_cbd,,
-0.75,0,1
0.00,110693,89135
0.75,0,8661


Péages négatifs ramenés à 0 : 1

Valeurs de frais_cbd par année APRÈS correction :


date_debut,2024,2025
frais_cbd,,
0.00,110693,89136
0.75,0,8661


Nombre de lignes : 208,490 (inchangé)


Elle ramène à 0 les péages négatifs, qui sont des annulations.
Effet sur les données : il ne reste que des montants positifs ou nuls.

In [15]:
# ---------- Cellule 11 : normalisation des montants CBD inhabituels ----------
TARIF_CBD_TAXI = 0.75

inhabituel = (green["frais_cbd"] > 0) & (green["frais_cbd"] != TARIF_CBD_TAXI)
print(f"Montants CBD inhabituels : {int(inhabituel.sum()):,}")
display(green.loc[inhabituel, "frais_cbd"].value_counts().to_frame("nb_courses"))

green.loc[inhabituel, "frais_cbd"] = TARIF_CBD_TAXI
print("Valeurs de frais_cbd après normalisation :")
display(green["frais_cbd"].value_counts().sort_index().to_frame("nb_courses"))

Montants CBD inhabituels : 0


,nb_courses
frais_cbd,


Valeurs de frais_cbd après normalisation :


,nb_courses
frais_cbd,
0.00,199829
0.75,8661


Elle ramène à 0,75 $ les montants positifs anormaux, puisque c’est aussi le tarif des taxis verts.
Effet sur les données : il ne reste que 0 ou 0,75 $.

In [16]:
# ---------- Cellule 12 : bilan du nettoyage de Green ----------
n_fin = len(green)
print(f"Lignes au départ  : {N_DEPART:,}")
print(f"Lignes gardées    : {n_fin:,} ({n_fin/N_DEPART:.2%})")
print(f"Lignes supprimées : {N_DEPART - n_fin:,} ({1 - n_fin/N_DEPART:.2%}) – doublons, hors période, zone de départ inconnue")

bilan = pd.DataFrame({
    "type":      green.dtypes.astype(str),
    "nb_vides":  green.isna().sum(),
    "pct_vides": (green.isna().mean() * 100).round(3),
})
print("\nValeurs vides par colonne (mesures impossibles vidées, courses gardées) :")
display(bilan)

print("Statistiques finales des colonnes numériques :")
display(green.describe(include="number", percentiles=[0.01, 0.5, 0.99]).T.round(2))

print(f"Période : du {green['date_debut'].min()} au {green['date_debut'].max()}")
print(f"Zones de départ distinctes : {green['zone_depart'].nunique()}")

Lignes au départ  : 209,194
Lignes gardées    : 208,490 (99.66%)
Lignes supprimées : 704 (0.34%) – doublons, hors période, zone de départ inconnue

Valeurs vides par colonne (mesures impossibles vidées, courses gardées) :


,type,nb_vides,pct_vides
date_debut,datetime64[ns],0,0.000
date_fin,datetime64[ns],0,0.000
zone_depart,int16,0,0.000
zone_arrivee,Int16,2011,0.965
distance,float32,10698,5.131
prix,float32,839,0.402
pourboire,float32,15,0.007
paiement,float32,0,0.000
frais_cbd,float32,0,0.000
duree_min,float32,7665,3.676


Statistiques finales des colonnes numériques :


,count,mean,std,min,1%,50%,99%,max
zone_depart,208490.0,95.660919,56.302225,1.0,7.0,75.0,260.0,263.0
zone_arrivee,206479.0,138.739029,75.977363,1.0,7.0,138.0,263.0,263.0
distance,197792.0,2.830513,2.986192,0.01,0.12,1.9,15.77,99.279999
prix,207651.0,17.245323,13.940505,0.01,3.0,13.5,70.0,470.600006
pourboire,208475.0,2.417248,3.113349,0.0,0.0,2.0,12.86,150.0
paiement,208490.0,1.491299,0.956552,1.0,1.0,1.0,5.0,5.0
frais_cbd,208490.0,0.031156,0.149665,0.0,0.0,0.0,0.75,0.75
duree_min,200825.0,14.689069,11.540382,1.0,2.1,11.9,59.916668,179.46666


Période : du 2024-02-01 00:00:54 au 2025-03-31 23:54:20
Zones de départ distinctes : 240


Elle résume le nettoyage : lignes gardées, valeurs vidées par colonne, statistiques finales et période.
Effet sur les données : aucun, c’est le contrôle final.

In [17]:
# ---------- Cellule 13 : sauvegarde de Green nettoyé (CSV compressé) ----------
libre = shutil.disk_usage(CLEAN_DIR).free / 1e9
print(f"Espace libre sur le disque : {libre:.2f} Go")

t0 = time.time()
cible = os.path.join(CLEAN_DIR, "green_clean.csv.gz")
green.to_csv(cible, index=False, chunksize=1_000_000, date_format="%Y-%m-%d %H:%M:%S",
             compression={"method": "gzip", "compresslevel": 1})

print(f"Green nettoyé sauvegardé : {cible}")
print(f"{len(green):,} lignes, {green.shape[1]} colonnes – {os.path.getsize(cible)/1e9:.2f} Go – {time.time()-t0:,.0f} s")
print("Colonnes :", list(green.columns))

Espace libre sur le disque : 2.73 Go
Green nettoyé sauvegardé : c:\Users\DELL\Desktop\esprit\ML\notebook_v0\Nootebook_vf\clean_data\green_clean.csv.gz
208,490 lignes, 10 colonnes – 0.00 Go – 2 s
Colonnes : ['date_debut', 'date_fin', 'zone_depart', 'zone_arrivee', 'distance', 'prix', 'pourboire', 'paiement', 'frais_cbd', 'duree_min']
